# Fetching Rosters for one Year

This is a helper notebook that reuses code from `data/igem_scrapping_2025/.ipynb_checkpoints/get_team_rosters-checkpoint.ipynb` to fetch teams and their rosters for one year (2025 originally here). The output is a TSV file with one row per team member. Names and IDs are not anonymized in this file. 

In [2]:
import requests
import pandas as pd
from tqdm import tqdm

API_ROOT = "https://api.igem.org/v1"

In [3]:
def fetch_teams_api(year):
    all_teams = []
    page = 1
    while True:
        url = f"https://api.igem.org/v1/teams"
        params = {"year": year, "pageSize": 100, "page": page}
        r = requests.get(url, params=params)
        if r.status_code != 200:
            print(f"{year} page {page} returned {r.status_code}")
            break
        try:
            data = r.json().get("data", [])
            if not data:
                break
            all_teams.extend(data)
            page += 1
        except Exception as e:
            print(f"JSON decode error: {e}")
            break
    return all_teams


In [4]:
teams_2025 = []
year = 2025

teams = fetch_teams_api(year)
print(f"{year}: {len(teams)} teams")
for team in teams:
    team["year"] = year
teams_2025.extend(teams)

print(f"Total teams retrieved: {len(teams_2025)}")

2025: 421 teams
Total teams retrieved: 421


In [5]:
teams_2025

[{'id': 5587,
  'name': 'Aachen',
  'region': 'europe',
  'country': 'DEU',
  'villageUUID': '3af5bd56-37fb-42ab-a17f-1068cde899a0',
  'kind': 'collegiate',
  'section': 'overgrad',
  'status': 'accepted',
  'year': 2025,
  'wikiURL': 'https://2025.igem.wiki/aachen',
  'isRemote': False,
  'ctoeUUID': None},
 {'id': 5936,
  'name': 'Aalto-Helsinki',
  'region': 'europe',
  'country': 'FIN',
  'villageUUID': 'f62341ec-70ee-4060-a2a3-7e7297e63e28',
  'kind': 'collegiate',
  'section': 'overgrad',
  'status': 'accepted',
  'year': 2025,
  'wikiURL': 'https://2025.igem.wiki/aalto-helsinki',
  'isRemote': False,
  'ctoeUUID': None},
 {'id': 5794,
  'name': 'ABOA',
  'region': 'europe',
  'country': 'FIN',
  'villageUUID': 'c3977ad0-ffb8-469b-93ea-c6ecf5ab185c',
  'kind': 'collegiate',
  'section': 'overgrad',
  'status': 'accepted',
  'year': 2025,
  'wikiURL': 'https://2025.igem.wiki/aboa',
  'isRemote': False,
  'ctoeUUID': None},
 {'id': 5999,
  'name': 'AEGIS',
  'region': 'asia',
  'co

In [ ]:
skipped_entries = []

def get_roster(team):
    year = team.get("year")
    team_id = team.get("id")
    team_name = team.get("name")
    url = f"https://api.igem.org/v1/teams/{team_id}/roster"

    try:
        r = requests.get(url)
        r.raise_for_status()
        data = r.json()
        records = []

        role_map = {
            "student": "Student Member",
            "student-leader": "Student Leader",
            "instructor": "Instructor",
            "primary-pi": "Primary PI",
            "secondary-pi": "Secondary PI"
        }

        for entry in data:
            roster_uuid = entry.get("uuid", "")
            role_key = entry.get("role", "").lower()
            role = role_map.get(role_key, role_key.replace("-", " ").title())
            member = entry.get("member") or {}
            user = member.get("user") or {}

            records.append({
                "Year": year,
                "TeamID": team_id,
                "Team": team_name,
                #"UserUUID": member.get("uuid", ""),
                "Username": member.get("username", ""),
                "FullName": user.get("publicName", ""),
                "Role": role,
                # "Institution": user.get("institution", ""),
                # "Title": user.get("title", ""),
                # "Affiliation": user.get("affiliation", ""),
                # "Country": user.get("country", ""),
                "RosterUUID": roster_uuid #the same as the teamRosterUUID in attribution-form
            })

        return records

    except Exception as e:
        print(f"Error for team {team_name} ({team_id}): {e}")
        skipped_entries.append({
            "Year": year,
            "TeamID": team_id,
            "Team": team_name,
            "Error": str(e),
            "RawEntry": None
        })
        return []


all_records = []
for team in tqdm(teams_2025, desc="Fetching rosters"):
    all_records.extend(get_roster(team))

df_roster = pd.DataFrame(all_records)
df_roster.sort_values(["Year", "Team", "Role"], inplace=True)
df_roster.to_csv("../data/igem_scrapping_2025/team_roster_2025_after_jamboree.tsv", sep="\t", index=False)
print(f"Saved team_roster_2025.tsv with {len(df_roster)} rows")

Fetching rosters: 100%|██████████| 421/421 [00:23<00:00, 18.23it/s]


Saved team_roster_2025.tsv with 10186 rows
